In [2]:
import ee
import geemap

# 1. Autenticación e Inicialización
# ee.Authenticate() # Descomenta la primera vez que lo uses
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project = "ee-my-usenvdi" )

# 2. Definir Área de Interés (AOI): Francia, España, Portugal, Andorra, Mónaco
paises = ['France', 'Spain', 'Portugal', 'Andorra', 'Monaco']
aoi = ee.FeatureCollection("FAO/GAUL/2015/level0") \
        .filter(ee.Filter.inList('ADM0_NAME', paises)) \
        .union(100) # Unimos los polígonos para evitar bordes internos

# 3. Parámetros Temporales (Últimos 5 años completos)
start_year = 2019
end_year = 2023
years = ee.List.sequence(start_year, end_year)
months = ee.List.sequence(1, 12)

# 4. Función para extraer la Temperatura Máxima Absoluta por mes
def get_monthly_max(year):
    def get_max_for_month(month):
        start_date = ee.Date.fromYMD(year, month, 1)
        end_date = start_date.advance(1, 'month')
        
        # Filtramos datos diarios para ese mes específico
        dataset = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
            .filterBounds(aoi) \
            .filterDate(start_date, end_date) \
            .select('temperature_2m')
            
        # Calculamos el máximo del mes, pasamos a Celsius y guardamos la fecha
        max_temp = dataset.max().subtract(273.15) \
            .set('system:time_start', start_date.millis()) \
            .clip(aoi)
        return max_temp
    
    return months.map(get_max_for_month)

# Ejecutamos la función y aplanamos la lista (obtenemos una colección de 60 imágenes)
monthly_max_list = years.map(get_monthly_max).flatten()
monthly_max_col = ee.ImageCollection.fromImages(monthly_max_list)

# 5. Calcular Media y Desviación Estándar de los 5 años
# Esto genera una imagen de la media y una de la desviación para cada píxel
mean_temp = monthly_max_col.mean()
std_temp = monthly_max_col.reduce(ee.Reducer.stdDev())

# 6. Calcular la Anomalía Estandarizada (Z-score) para cada mes
# Fórmula: Z = (Valor del mes - Media) / Desviación Estándar
def calculate_z_score(image):
    z_score = image.subtract(mean_temp).divide(std_temp) \
        .rename('Desviacion_Estandar') \
        .set('system:time_start', image.get('system:time_start'))
    return z_score

z_score_col = monthly_max_col.map(calculate_z_score)


In [3]:

# 7. Configuración del Mapa y Visualización Timelapse
Map = geemap.Map()
Map.centerObject(aoi, 5)

# Paleta divergente: Azul (frío extremo), Blanco (normal), Rojo oscuro (calor extremo)
# Los valores van de -3 a +3 desviaciones estándar (que cubre el 99.7% de los casos normales)
vis_params_z = {
    'min': -2.5,
    'max': 2.5,
    'palette': ['#313695', '#4575b4', '#74add1', '#abd9e9', '#ffffff', 
                '#fdae61', '#f46d43', '#d73027', '#a50026'] 
}

# Añadimos la herramienta de Timelapse de Geemap
# Nota: require un poco de tiempo para procesar los 60 meses en los servidores de Google
Map.add_time_slider(z_score_col, vis_params_z, layer_name="Anomalía de Temp. Máx (Z-Score)", time_interval=1.5)

# Mostramos el mapa
Map

Map(center=[43.24011481766895, -1.205174646458778], controls=(WidgetControl(options=['position', 'transparent_…